# CCK sweep 1_cap1000 (one sweep per notebook, run in parallel)

Clones pupa3066/context-configuration-builder at a pinned commit and runs benchmark/conceptual_fair_metrics.py unchanged,
so chunking (8-line chunks), BM25 tokenization, corpus loading and question mining are identical to the September N=240 cap sweep.

Earlier notebooks (cck_kaggle_sweep.ipynb, cck_kaggle_full_corroboration.ipynb) used an inline 400-word chunker and a different BM25 tokenizer; their results are NOT comparable to the committed study.

Internet must be ON. CPU is enough. Each sweep writes its JSON as soon as it finishes.

In [ ]:
!pip install fastembed transformers --quiet
print('deps installed')

In [ ]:
# Cache every network dependency first so a later network drop cannot kill a sweep.
from fastembed import TextEmbedding
from transformers import AutoTokenizer
for m in ['BAAI/bge-small-en-v1.5', 'BAAI/bge-base-en-v1.5']:
    print('caching', m, flush=True); TextEmbedding(m)
AutoTokenizer.from_pretrained('unsloth/gemma-2-2b-bnb-4bit')  # imported by compare_algorithms.py
print('all cached', flush=True)

In [ ]:
import subprocess, glob
SHA = 'b8cc6d88e8dae49a31bb6a5ea84e87332dd79615'
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(c, '->', r.returncode, r.stderr.strip()[-300:])
sh('git clone https://github.com/pupa3066/context-configuration-builder.git /tmp/cck')
sh(f'cd /tmp/cck && git checkout {SHA}')
# depth=200 keeps every .py file (verified 37/91/422) and is much faster than a full clone
for name, url in [('requests','https://github.com/psf/requests.git'),
                  ('click','https://github.com/pallets/click.git'),
                  ('black','https://github.com/psf/black.git')]:
    sh(f'git clone --depth=200 {url} /tmp/{name}')
    n = len(glob.glob(f'/tmp/{name}/**/*.py', recursive=True))
    head = subprocess.run(f'git -C /tmp/{name} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
    print(f'{name}: {n} .py files, HEAD {head}')

In [ ]:
import subprocess, sys
REPOS = 'requests:/tmp/requests click:/tmp/click black:/tmp/black'
COMMON = f'--repos {REPOS} --n-per-repo 80 --k 6 --bootstrap 1000'
SWEEPS = [
    ('SWEEP_cap_1000.json', '--max-chunks 1000 --embed-model BAAI/bge-small-en-v1.5 --sem-threshold 0.72'),
]
import time
for out, extra in SWEEPS:
    cmd = f'cd /tmp/cck && {sys.executable} -u benchmark/conceptual_fair_metrics.py {COMMON} {extra} --out /kaggle/working/{out}'
    print('\n===', out, time.strftime('%H:%M:%S'), flush=True)
    # stream output live so progress is visible
    with subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as pr:
        for line in pr.stdout:
            if 'SyntaxWarning' in line or line.strip().startswith('<unknown>'): continue
            print(line, end='', flush=True)
    print('done', out, time.strftime('%H:%M:%S'), 'exit', pr.returncode, flush=True)


In [ ]:
import json, os
SEPT = {'m1_substring': 0.8667, 'm2_defline': 0.8208, 'm3_sem_credit': 0.9583}  # September BM25, CURRENT_STATE.md
for out, _ in SWEEPS:
    p = f'/kaggle/working/{out}'
    if not os.path.exists(p): print(out, 'MISSING'); continue
    d = json.load(open(p)); print(f"\n{out}  N={d['N']}  model={d['embed_model']}  cap={d['max_chunks']}  thresh={d['sem_threshold']}")
    for m in ('m1_substring','m2_defline','m3_sem_credit'):
        b = d['arms']['bm25'][m]; s = d['arms']['semantic'][m]
        ov = b[1] <= s[2] and s[1] <= b[2]
        print(f'  {m:<15} BM25={b[0]} {b[1:]}  sem={s[0]} {s[1:]}  overlap={ov}  BM25-vs-Sept={round(b[0]-SEPT[m],4)}')
print('\nIf BM25-vs-Sept is not ~0 the repos changed since September; report both runs, do not mix them.')